In [2]:
import streamlit as st
import numpy as np
from PIL import Image
from ultralytics import YOLO
import cv2
import easyocr
from util import set_background
from transformers import AutoModel, AutoTokenizer
import pandas as pd

set_background("./imgs/background.png")


tokenizer = AutoTokenizer.from_pretrained("ucaslcl/GOT-OCR2_0", trust_remote_code=True)
ocr_model = AutoModel.from_pretrained(
    "ucaslcl/GOT-OCR2_0",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    device_map="cuda",
    use_safetensors=True,
    pad_token_id=tokenizer.eos_token_id,
)
ocr_model = ocr_model.eval().cuda()

detect_car_model = YOLO(
    "/home/duckq1u/Documents/DoAnChuyenNganh/DACS4/models/yolov8-car-detect.pt"
)
license_plate_model = YOLO(
    "/home/duckq1u/Documents/DoAnChuyenNganh/DACS4/models/yolov8-car-detect-license-plate-detect.pt"
)
reader = easyocr.Reader(["en"], gpu=True)
vehicles = [2]


header = st.container()
body = st.container()


threshold = 0.15

state = "Uploader"

if "state" not in st.session_state:
    st.session_state["state"] = "Uploader"


# NOTE: detect id vehicles
def model_prediction(img):
    license_plate_list = []
    car_list = []
    license_number_list = []
    car_detect = detect_car_model(img)[0]

    for box in car_detect.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])  # Extract coordinates
        confidence = box.conf[0].item()
        label = f"Car {confidence:.2f}"
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)  # Green box for cars
        cv2.putText(
            img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2
        )
        car_crop = img[y1:y2, x1:x2]
        result = license_plate_model(car_crop)[0]
        license_number = ocr_model.chat(tokenizer, result, ocr_type="ocr")

        license_plate_list.append(result)
        car_list.append(car_crop)
        license_number_list.append(license_number)

    return {"license_plate_list": license_plate_list, "car_list": car_list, "license_number_list": license_number_list}


def change_state_uploader():
    st.session_state["state"] = "Uploader"


def change_state_camera():
    st.session_state["state"] = "Camera"


def change_state_live():
    st.session_state["state"] = "Live"


with header:
    _, col1, _ = st.columns([0.2, 1, 0.1])
    col1.title("💥 License Car Plate Detection 🚗")

    _, col0, _ = st.columns([0.15, 1, 0.1])
    col0.image("./imgs/test_background.jpg", width=500)

    _, col4, _ = st.columns([0.1, 1, 0.2])
    col4.subheader("Computer Vision Detection with YoloV8 🧪")

    _, col, _ = st.columns([0.3, 1, 0.1])
    col.image("./imgs/plate_test.jpg")

    _, col5, _ = st.columns([0.05, 1, 0.1])

    st.write(
        "The differents models detect the car and the license plate in a given image, then extracts the info about the license using EasyOCR, and crop and save the license plate as a Image, with a CSV file with all the data.   "
    )


with body:
    _, col1, _ = st.columns([0.1, 1, 0.2])
    col1.subheader("Check It-out the License Car Plate Detection Model 🔎!")

    _, colb1, colb2, colb3 = st.columns([0.2, 0.7, 0.6, 1])

    if st.session_state["state"] == "Uploader":
        img = st.file_uploader("Upload a Car Image: ", type=["png", "jpg", "jpeg"])
    elif st.session_state["state"] == "Camera":
        img = st.camera_input("Take a Photo: ")
    elif st.session_state["state"] == "Live":
        # webrtc_streamer(key="sample", video_processor_factory=VideoProcessor)
        img = None

    _, col2, _ = st.columns([0.3, 1, 0.2])

    _, col5, _ = st.columns([0.8, 1, 0.2])

    if img is not None:
        image = np.array(Image.open(img))
        col2.image(image)
        results = model_prediction(image)
        df = pd.DataFrame(
            {
                "Car Image": results["car_list"],
                "License Plate": results["license_plate_list"],
                "License Number": results["license_number_list"],
            }
        )
        st.table(df)
        print(results)


2025-05-05 13:03:56.936 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-05 13:03:56.937 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-05 13:04:04.517 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-05 13:04:04.517 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-05 13:04:04.518 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-05 13:04:04.518 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-05 13:04:04.518 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-05 13:04:04.519 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar